In [1]:
import pandas as pd
import numpy as np

In [4]:
df_fe = pd.read_csv("../raw_data/features_enriched_data.csv")

In [ ]:
# =========================
# YEARS OPERATING
# =========================
reference_date = pd.Timestamp.today().normalize()

df_fe["years_operating"] = (
    (reference_date - df_fe["founded_at"]).dt.days / 365.25
)

# =========================
# TARGET 1:
# acquired + operating vs closed
# =========================
df_fe["target_bin_acq_oper_vs_closed"] = df_fe["status_enriched"].map({
    "acquired": 1,
    "operating": 1,
    "closed": 0
})

# =========================
# TARGET 2:
# acquired vs closed
# =========================
df_fe["target_bin_acq_vs_closed"] = df_fe["status_enriched"].map({
    "acquired": 1,
    "closed": 0
})

# =========================
# TARGET 3:
# multiclass
# =========================
df_fe["target_multiclass"] = df_fe["status_enriched"].map({
    "closed": 0,
    "operating": 1,
    "acquired": 2
})

# =========================
# TARGET 4:
# refined target
# =========================
def classify_status(row):
    if row["status_enriched"] == "acquired":
        return 1

    elif row["status_enriched"] == "operating":
        if pd.isna(row["years_operating"]):
            return np.nan

        if row["years_operating"] > 4:
            return 1

        elif row["years_operating"] <= 4:
            if row.get("category_total") in ["high", "medium-high"]:
                return 1
            else:
                return 0

    elif row["status_enriched"] == "closed":
        return 0

    return np.nan

df_fe["target_bin_refined"] = df_fe.apply(classify_status, axis=1)

# =========================
# FINAL CHECKS
# =========================
target_cols = [
    "target_bin_acq_oper_vs_closed",
    "target_bin_acq_vs_closed",
    "target_multiclass",
    "target_bin_refined"
]

for col in target_cols:
    print(f"\n{col}")
    print(df_fe[col].value_counts(dropna=False, normalize=True))


target_bin_acq_oper_vs_closed
target_bin_acq_oper_vs_closed
1.0    0.725402
0.0    0.248653
NaN    0.025945
Name: proportion, dtype: float64

target_bin_acq_vs_closed
target_bin_acq_vs_closed
NaN    0.687803
0.0    0.248653
1.0    0.063544
Name: proportion, dtype: float64

target_multiclass
target_multiclass
1.0    0.661858
0.0    0.248653
2.0    0.063544
NaN    0.025945
Name: proportion, dtype: float64

target_bin_refined
target_bin_refined
1.0    0.725402
0.0    0.248653
NaN    0.025945
Name: proportion, dtype: float64


In [6]:
df_fe.columns

Index(['permalink', 'category_list', 'market', 'funding_total_usd',
       'country_code', 'state_code', 'region', 'funding_rounds', 'founded_at',
       'first_funding_at', 'last_funding_at', 'seed', 'venture',
       'debt_financing', 'angel', 'grant', 'private_equity', 'round_A',
       'round_B', 'round_C', 'round_D', 'round_E', 'status_enriched',
       'avg_raised_per_round', 'age_first_funding_days', 'has_multiple_rounds',
       'funding_span_days', 'avg_years_between_rounds', 'region_group',
       'market_clean', 'industry_group', 'years_operating'],
      dtype='object')

In [8]:
df_fe.to_csv("features_enriched_with_targets.csv", index=False)
